# AI Experiment Prioritization — IJAH Analytics — Kelompok 7

**Proof-of-Concept**: AI Experiment Prioritization untuk IJAH Analytics Menguji apakah pendekatan menggabungkan confidence, jumlah bukti, novelty, dan potensi sinergi jadi satu skor prioritas eksperimen bisa berjalan dan menghasilkan urutan yang masuk akal.

Karena kelompok 7 belum punya akses ke data BLM-NII/NIMS asli IJAH, PoC ini memakai data sintetis yang strukturnya meniru fitur yang sudah didefinisikan di LKP P4 (confidence_score, source_count, interaction_count, com_similarity_simcomp, pro_similarity_smithwaterman), ditambah nims_synergy_score.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

np.random.seed(42)
N = 500  # jumlah pasangan senyawa-protein dummy

# 1. Bangkitkan data sintetis meniru struktur fitur IJAH
confidence_score = np.random.beta(2, 2, N)                     # skor prediksi BLM-NII (0-1)
source_count = np.random.poisson(1.5, N)                        # jumlah database yang mengonfirmasi relasi
interaction_count = np.random.exponential(3, N)                 # jumlah interaksi tercatat pada node
com_similarity = np.random.beta(2, 3, N)                        # kemiripan struktur kimia antar-senyawa
pro_similarity = np.random.beta(2, 3, N)                        # kemiripan sekuens antar-protein
nims_synergy_score = np.random.beta(2, 2, N)                    # skor potensi sinergi dari NIMS

df = pd.DataFrame({
    "compound_id": [f"C{i:04d}" for i in range(N)],
    "protein_id": [f"P{i:04d}" for i in range(N)],
    "confidence_score": confidence_score,
    "source_count": source_count,
    "interaction_count": interaction_count,
    "com_similarity": com_similarity,
    "pro_similarity": pro_similarity,
    "nims_synergy_score": nims_synergy_score,
})

# 2. Hitung novelty_score, makin sedikit interaksi tercatat sebelumnya, makin novel
df["novelty_score"] = 1 / (1 + df["interaction_count"])

# 3. Normalisasi source_count ke rentang 0-1
df["source_count_norm"] = (df["source_count"] - df["source_count"].min()) / \
                           (df["source_count"].max() - df["source_count"].min())

# 4. Skor gabungan AI Experiment Prioritization (weighted sum)
W_CONFIDENCE = 0.4
W_EVIDENCE = 0.2
W_NOVELTY = 0.2
W_SYNERGY = 0.2

df["priority_score"] = (
    W_CONFIDENCE * df["confidence_score"] +
    W_EVIDENCE * df["source_count_norm"] +
    W_NOVELTY * df["novelty_score"] +
    W_SYNERGY * df["nims_synergy_score"]
)

# 5. Ground truth sintetis untuk evaluasi PoC
latent_quality = (
    0.5 * df["confidence_score"] +
    0.3 * df["nims_synergy_score"] +
    0.2 * df["novelty_score"] +
    np.random.normal(0, 0.12, N)
)
threshold = np.quantile(latent_quality, 0.80)
df["true_high_priority"] = (latent_quality >= threshold).astype(int)

# 6. Evaluasi: Average Precision (analog AUPR) dan Precision@10
ap_score = average_precision_score(df["true_high_priority"], df["priority_score"])

top10 = df.sort_values("priority_score", ascending=False).head(10)
precision_at_10 = top10["true_high_priority"].sum() / 10

print("=== HASIL PROOF-OF-CONCEPT: AI Experiment Prioritization ===")
print(f"Jumlah kandidat dummy diuji : {N}")
print(f"Average Precision (mirip AUPR)  : {ap_score:.3f}")
print(f"Precision@10 (dari 10 kandidat teratas): {precision_at_10:.2f}")
print()
print("Contoh 5 kandidat dengan prioritas tertinggi:")
print(top10[["compound_id", "protein_id", "confidence_score",
             "source_count", "novelty_score", "nims_synergy_score",
             "priority_score"]].head(5).round(3).to_string(index=False))

=== HASIL PROOF-OF-CONCEPT: AI Experiment Prioritization ===
Jumlah kandidat dummy diuji : 500
Average Precision (mirip AUPR)  : 0.608
Precision@10 (dari 10 kandidat teratas): 0.80

Contoh 5 kandidat dengan prioritas tertinggi:
compound_id protein_id  confidence_score  source_count  novelty_score  nims_synergy_score  priority_score
      C0083      P0083             0.899             1          0.998               0.676           0.723
      C0266      P0266             0.795             4          0.860               0.568           0.718
      C0456      P0456             0.814             4          0.654               0.644           0.700
      C0221      P0221             0.881             4          0.329               0.797           0.692
      C0185      P0185             0.780             3          0.652               0.778           0.684
